## 일별 박스오피스 현황에 대한 정보 수집
- 영화 진흥위원회에서 관리하는 api를 이용

### 1. 라이브러리 호출

In [22]:
import math
import time
from getpass import getpass                     # 인증키가 화면에 표시되지 않도록 하는 라이브러리
from urllib.parse import unquote                # URL을 보기 좋게 만들어주는 라이브러리

import requests
import pandas as pd

### 2. 인증키 입력

In [23]:
general_key = getpass("공공데이터포털 인증키를 입력하세요: ").strip()

SERVICE_KEY = unquote(general_key)

## 3. API 주소와 기본 요청 변수 설정

In [24]:
API_URL = ("http://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json")

# 한 번 호출할 때 가져오는 데이터 수
NUM_OF_ROWS = 10
TARGET_DT = 20260809


In [35]:
target_dt_list = list(range(20260727, 20260732)) + list(range(20260801, 20260810))
print(target_dt_list)

[20260727, 20260728, 20260729, 20260730, 20260731, 20260801, 20260802, 20260803, 20260804, 20260805, 20260806, 20260807, 20260808, 20260809]


## 4. 날짜 하나 가져와서 연습해보기

In [25]:
params = {
    "key" : SERVICE_KEY,
    "itemPerPage" : NUM_OF_ROWS,
    "targetDt" : TARGET_DT
}

response = requests.get(API_URL, params = params, timeout = 30)

In [26]:
print("HTTP 상태코드: ", response.status_code)
print("응답 형식: ", response.headers.get("Content-Type"))

HTTP 상태코드:  200
응답 형식:  application/json;charset=utf-8


In [27]:
api_data = response.json()

In [28]:
api_data

{'boxOfficeResult': {'boxofficeType': '일별 박스오피스',
  'showRange': '20260809~20260809',
  'dailyBoxOfficeList': [{'rnum': '1',
    'rank': '1',
    'rankInten': '0',
    'rankOldAndNew': 'OLD',
    'movieCd': '20250654',
    'movieNm': '오디세이',
    'openDt': '2026-08-05',
    'salesAmt': '5421127260',
    'salesShare': '44.9',
    'salesInten': '-327572430',
    'salesChange': '-5.7',
    'salesAcc': '20498854930',
    'audiCnt': '497554',
    'audiInten': '-28103',
    'audiChange': '-5.3',
    'audiAcc': '1874819',
    'scrnCnt': '1798',
    'showCnt': '5118'},
   {'rnum': '2',
    'rank': '2',
    'rankInten': '0',
    'rankOldAndNew': 'OLD',
    'movieCd': '20262770',
    'movieNm': '스파이더맨: 브랜드 뉴 데이',
    'openDt': '2026-07-29',
    'salesAmt': '4782490470',
    'salesShare': '39.6',
    'salesInten': '-559568450',
    'salesChange': '-10.5',
    'salesAcc': '61881618240',
    'audiCnt': '463917',
    'audiInten': '-50220',
    'audiChange': '-9.8',
    'audiAcc': '5916397',
    'scrn

In [29]:
# 1. 중간 딕셔너리 먼저 꺼내기
box_office = api_data.get("boxOfficeResult", {})

# 2. 각 항목 꺼내기
target_date = box_office.get("showRange", "")
daily_list = box_office.get("dailyBoxOfficeList", [])

print("조회 날짜:", target_date)
print("영화 목록 수:", len(daily_list))

조회 날짜: 20260809~20260809
영화 목록 수: 10


- 공공데이터 api 쓸때랑 형태가 다르네
- 확실히 json형태에서 파이썬의 딕셔너리와 유사한 형태로 변환한 후 데이터가 어떻게 축적되어있나 확인 후 필요한 부분들을 가져와주면 되네
- daily_list에 현재 영화 하나에 대한 정보가 쫘라라라ㅏㄱ 있을 텐데 각각의 순위만 따로 가져가고, 영화 이름만 가져가고 싶으면 for문을 이용해서 각각 가져간 후 데이터 프레임에 넣는 방법이 당장 생각나는 방법
- 디벨롭 할 수 있는 방안으로는 강사님이 해주셨던 거 차용해서 생각해보기

In [32]:
daily_list[1].values()

dict_values(['2', '2', '0', 'OLD', '20262770', '스파이더맨: 브랜드 뉴 데이', '2026-07-29', '4782490470', '39.6', '-559568450', '-10.5', '61881618240', '463917', '-50220', '-9.8', '5916397', '1763', '6052'])

In [42]:
date_list = []
daily_rank = []

for date in target_dt_list:
    params = {
    "key" : SERVICE_KEY,
    "itemPerPage" : NUM_OF_ROWS,
    "targetDt" : TARGET_DT
    }
    response = requests.get(API_URL, params = params, timeout = 30)
    api_data = response.json()
    box_office = api_data.get("boxOfficeResult", {})
    target_date = box_office.get("showRange", "")
    daily_list = box_office.get("dailyBoxOfficeList", [])
    for i in range(0, 10):
        date_list.append(target_date)
        daily_rank.append(list(daily_list[i].values())[1])      # 한 문장으로 줄일 수 있네
        
print(date_list[:3])
print(daily_rank[:3])

['20260809~20260809', '20260809~20260809', '20260809~20260809']
['1', '2', '3']


In [39]:
print(len(date_list))
print(len(daily_rank))

140
140
